# Module 45 — Uplift Modeling: T-Learner with PyTorch

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: PyTorch, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 44 (Uplift Two-Model)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Data](#3-synthetic-data)
4. [T-Learner Architecture](#4-t-learner-architecture)
5. [Training](#5-training)
6. [Evaluation](#6-evaluation)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why T-Learner for uplift?

T-Learner uses **separate neural networks** for treatment and control:
- **Flexible**: Can capture complex non-linear relationships.
- **Deep learning**: Better for high-dimensional features.
- **Calibration**: Better probability estimates.

**Applications**:
1. **Personalization**: Tailor experiences by uplift segment.
2. **Campaign optimization**: Maximize incremental conversions.
3. **Resource allocation**: Focus on high-uplift customers.

In this notebook we will:
1. Generate synthetic campaign data.
2. Build T-Learner with PyTorch neural networks.
3. Train treatment and control models.
4. Compare with sklearn-based approach.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Deep learning ────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ All imports loaded")
print(f"  Device: {device}")

---
## §3 · Synthetic Data

In [ ]:
# Generate synthetic campaign data
N_CUSTOMERS = 15000

data = {
    'age': np.random.normal(40, 12, N_CUSTOMERS).clip(18, 80).astype(int),
    'income': np.random.lognormal(10.5, 0.8, N_CUSTOMERS),
    'tenure_months': np.random.exponential(24, N_CUSTOMERS).astype(int),
    'previous_purchases': np.random.poisson(5, N_CUSTOMERS),
    'avg_order_value': np.random.exponential(75, N_CUSTOMERS),
    'days_since_last_purchase': np.random.exponential(30, N_CUSTOMERS).astype(int),
}

df = pd.DataFrame(data)
df['treatment'] = np.random.binomial(1, 0.5, N_CUSTOMERS)

# Response with heterogeneous treatment effect
base_response = 0.1
treatment_effect = (
    0.15 * (df['previous_purchases'] > 3).astype(float) +
    0.10 * (df['days_since_last_purchase'] < 30).astype(float) +
    -0.05 * (df['age'] > 60).astype(float)
)

response_prob = base_response + df['treatment'] * treatment_effect
response_prob = response_prob.clip(0.01, 0.99)
df['converted'] = np.random.binomial(1, response_prob)

print(f"✓ Generated {N_CUSTOMERS:,} customer records")
print(f"  Treatment conversion: {df[df['treatment']==1]['converted'].mean()*100:.2f}%")
print(f"  Control conversion: {df[df['treatment']==0]['converted'].mean()*100:.2f}%")

---
## §4 · T-Learner Architecture

In [ ]:
# Define neural network for uplift
class UpliftNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# Create separate models for treatment and control
feature_cols = ['age', 'income', 'tenure_months', 'previous_purchases',
                'avg_order_value', 'days_since_last_purchase']

model_treatment = UpliftNetwork(len(feature_cols)).to(device)
model_control = UpliftNetwork(len(feature_cols)).to(device)

print(f"✓ T-Learner models created")
print(f"  Treatment model parameters: {sum(p.numel() for p in model_treatment.parameters()):,}")
print(f"  Control model parameters: {sum(p.numel() for p in model_control.parameters()):,}")

---
## §5 · Training

In [ ]:
# Prepare data
X = df[feature_cols].values
y = df['converted'].values
treatment = df['treatment'].values

# Split
X_train, X_test, y_train, y_test, treat_train, treat_test = train_test_split(
    X, y, treatment, test_size=0.3, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training setup
criterion = nn.BCELoss()
optimizer_t = torch.optim.Adam(model_treatment.parameters(), lr=1e-3)
optimizer_c = torch.optim.Adam(model_control.parameters(), lr=1e-3)

N_EPOCHS = 20
BATCH_SIZE = 256

print(f"Training setup:")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training samples: {len(y_train):,}")

---
## §6 · Evaluation

In [ ]:
# Evaluate (using random models for demo)
model_treatment.eval()
model_control.eval()

with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test_scaled).to(device)
    p_treatment = model_treatment(X_test_t).cpu().numpy().flatten()
    p_control = model_control(X_test_t).cpu().numpy().flatten()

# Calculate uplift
uplift = p_treatment - p_control

print(f"✓ Uplift predictions complete")
print(f"  Mean uplift: {uplift.mean():.4f}")
print(f"  Positive uplift: {(uplift > 0).sum():,} ({(uplift > 0).mean()*100:.1f}%)")
print(f"\nNote: These are random model predictions.")
print(f"After training, uplift would show meaningful separation.")

---
## §7 · Visualization

In [ ]:
# Visualize T-Learner architecture
print("T-Learner Architecture:")
print("=" * 70)
print("""
┌─────────────────────────────────────────────────────────────┐
│                    T-Learner Model                          │
├─────────────────────────────────────────────────────────────┤
│  Input Features                                             │
│  ┌──────────────────────────────────────────────────────┐  │
│  │ [age, income, tenure, purchases, avg_order, days]    │  │
│  └──────────────────────────────────────────────────────┘  │
│         │                                    │              │
│         ▼                                    ▼              │
│  ┌──────────────┐                    ┌──────────────┐      │
│  │   Treatment   │                    │   Control     │      │
│  │   Model       │                    │   Model       │      │
│  │   (Neural Net)│                    │   (Neural Net)│      │
│  └──────┬───────┘                    └──────┬───────┘      │
│         │                                    │              │
│         ▼                                    ▼              │
│  ┌──────────────┐                    ┌──────────────┐      │
│  │ P(convert|T)  │                    │ P(convert|C)  │      │
│  └──────┬───────┘                    └──────┬───────┘      │
│         │                                    │              │
│         └────────────┬───────────────────────┘              │
│                      ▼                                      │
│              ┌──────────────┐                               │
│              │    Uplift     │                               │
│              │ = P(T) - P(C) │                               │
│              └──────────────┘                               │
└─────────────────────────────────────────────────────────────┘
""")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for ML Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | High-dimensional data, complex treatment effects |
> | **Architecture** | T-Learner with neural networks |
> | **Training** | Separate models for treatment and control |
> | **Evaluation** | Qini curve, AUUC, calibration |
> | **Deployment** | Score real-time, A/B test against baseline |
>
> ### 💡 Model Comparison
>
> | Model | Pros | Cons | Best For |
> |-------|------|------|----------|
> | Two-Model (Sklearn) | Simple | Limited complexity | Baseline |
> | T-Learner (PyTorch) | Flexible | Requires more data | Complex patterns |
> | X-Learner | Better for small samples | Complex | Imbalanced data |
>
> ### 🔑 Key Takeaways
>
> 1. **T-Learner uses separate networks** for treatment and control.
> 2. **Neural networks capture complex patterns** better than linear models.
> 3. **Uplift = P(Treatment) - P(Control)** for each customer.
> 4. **PyTorch provides flexibility** for custom architectures.
> 5. **A/B test uplift models** against standard targeting.